# Web scraping de productos de belleza – Cromantic

**Objetivo:** extraer información de productos (referencia, marca, nombre y precio) de la tienda en línea Cromantic, consolidarla en un DataFrame, exportarla a CSV y hacer un análisis descriptivo básico.

**Fuente:** https://www.cromantic.com/

## 1. Importación de librerías

Se importan todas las librerías una sola vez al inicio del notebook.

In [ ]:
# Importar librerías
import requests
from bs4 import BeautifulSoup
import json
import html
import time
import pandas as pd
from google.colab import files

## 2. Conexión al sitio

Se hace una petición GET a la página principal. Se envía un `User-Agent` de navegador común para que el sitio responda como a una visita normal.

In [ ]:
# URL de la tienda Cromantic (sin parámetros de seguimiento de anuncios)
url = 'https://www.cromantic.com/'

# Cabeceras para simular una visita desde un navegador web común (se reutilizan en todo el notebook)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Realizar la petición GET
response = requests.get(url, headers=headers, timeout=10)

# Verificar si la conexión fue exitosa
if response.status_code == 200:
    print("¡Conexión exitosa! Estado:", response.status_code)
    # Parsear el contenido HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    # Mostrar el título de la página como prueba
    print("Título de la página:", soup.title.string.strip() if soup.title else "Sin título")
else:
    print(f"No se pudo acceder de forma directa. Código de estado: {response.status_code}")
    print("Nota: Algunas tiendas grandes usan sistemas de protección como Cloudflare que pueden requerir herramientas adicionales.")

¡Conexión exitosa! Estado: 200
Título de la página: Productos y Accesorios de Belleza | Cromantic


## 3. Función de extracción de datos estructurados (JSON-LD)

Muchas tiendas en línea publican la información de sus productos en bloques `<script type="application/ld+json">`. Esta función lee esos bloques y devuelve una lista de productos.

Se define una sola vez y se reutiliza en la página principal y en la paginación, para no repetir código. Contempla dos estructuras:
- **ItemList:** listado de varios productos.
- **Product:** un producto individual.

In [ ]:
MARCA_POR_DEFECTO = 'Cromantic'


def obtener_marca(producto):
    """Devuelve el nombre de la marca, ya venga como diccionario o como texto."""
    marca = producto.get('brand', {})
    if isinstance(marca, dict):
        return marca.get('name', MARCA_POR_DEFECTO)
    if isinstance(marca, str) and marca.strip():
        return marca.strip()
    return MARCA_POR_DEFECTO


def obtener_precio(producto):
    """Devuelve el precio como número (float) o None si no está disponible."""
    ofertas = producto.get('offers', {})
    if isinstance(ofertas, dict):
        precio_num = ofertas.get('highPrice') or ofertas.get('lowPrice') or ofertas.get('price')
        if precio_num:
            return float(precio_num)
    return None


def armar_registro(producto, caracteristicas):
    """Construye el diccionario de un producto con las columnas finales."""
    nombre = html.unescape(producto.get('name', 'No disponible'))  # corrige &apos; &amp; etc.
    sku = producto.get('sku', producto.get('id', 'No disponible'))
    marca = html.unescape(obtener_marca(producto))
    precio = obtener_precio(producto)

    return {
        "Referencia (SKU)": sku,
        "Nombre/Marca": f"[{marca}] {nombre}",
        "Valor (COP)": precio,  # numérico, para análisis
        "Valor": f"${precio:,.0f}".replace(",", ".") if precio is not None else "No disponible",  # texto para mostrar
        "Características": caracteristicas
    }


def extraer_productos(soup, caracteristicas):
    """Recorre los bloques JSON-LD de una página y devuelve la lista de productos encontrados."""
    productos = []
    for script in soup.find_all('script', type='application/ld+json'):
        if not script.string:
            continue
        try:
            datos = json.loads(script.string)

            # Si el JSON contiene una lista de elementos (ItemList)
            if datos.get('@type') == 'ItemList' and 'itemListElement' in datos:
                for item_wrapper in datos['itemListElement']:
                    producto = item_wrapper.get('item', {})
                    if producto:
                        productos.append(armar_registro(producto, caracteristicas))

            # Si es un objeto de producto único (Individual)
            elif datos.get('@type') == 'Product':
                productos.append(armar_registro(datos, caracteristicas))

        except Exception as e:
            print(f"Error al procesar un bloque JSON-LD: {e}")
    return productos

## 4. Extracción de productos de la página principal

Primero se intenta con los datos estructurados (JSON-LD). Si no se encuentra nada, se usa un plan B que lee las clases HTML de las tarjetas de producto.

In [ ]:
# Extracción con datos estructurados
productos_lista = extraer_productos(soup, "Belleza")

# Plan B: si no hay datos estructurados válidos, analizamos las clases HTML
if not productos_lista:
    for prod in soup.select('[class*="product"], [class*="card"], [class*="summary"]'):
        link = prod.find('a', href=True)
        if link and ('products' in link['href'] or 'producto' in link['href'] or 'frontpage' in link['href']):
            nombre_txt = link.text.strip().split('\n')[0].strip()  # Limpiamos el texto
            if nombre_txt and len(nombre_txt) > 5:

                # Buscar el precio en la clase de oferta o, si no hay, en la de precio regular
                precio_elem = prod.find(class_="price-item price-item--sale price-item--last")
                if not precio_elem:
                    precio_elem = prod.find(class_="price-item price-item--regular")

                precio_detectado = precio_elem.text.strip() if precio_elem else "Consultar precio dinámico"

                productos_lista.append({
                    "Referencia (SKU)": "No disponible",
                    "Nombre/Marca": f"[{MARCA_POR_DEFECTO}] {html.unescape(nombre_txt)}",
                    "Valor (COP)": None,
                    "Valor": precio_detectado,
                    "Características": "Belleza"
                })

# Creamos el DataFrame con los datos extraídos
if productos_lista:
    df_producto = pd.DataFrame(productos_lista)
    df_producto = df_producto.drop_duplicates(subset=['Nombre/Marca'])
    display(df_producto)
else:
    print("No se encontraron productos con las reglas aplicadas.")

,Referencia (SKU),Nombre/Marca,Valor (COP),Valor,Características
0,11028,[Coraje by Lina Tejeiro] Pestañina Coraje by L...,59990.0,$59.990,Belleza
1,11015,[Everyday Love] Shampoo Everyday Love Miracle ...,49900.0,$49.900,Belleza
2,11009,[Dr.Althea] Protector Solar Dr. Althea Green Tea,93990.0,$93.990,Belleza
3,11001,[Anyeluz] Acondicionador Afro Curly Anyeluz 600ml,49990.0,$49.990,Belleza
4,11000,[Anyeluz] Mascarilla Terapia Romero Anyeluz 300ml,42990.0,$42.990,Belleza
5,11006,[Loreal Professionnel] Kit Duo Mini L'Oréal Pr...,99990.0,$99.990,Belleza
6,11008,[Revox] Kit Revox Rutina Facial Perfecta,93990.0,$93.990,Belleza
7,11007,[Revox] Serum Revox Acido Glicolico 20%,41990.0,$41.990,Belleza
8,10997,[Anyeluz] Shampoo Afro Curly Anyeluz 600ml,49990.0,$49.990,Belleza
9,10993,[Medicube] Age-R Booster Pro X2 Medicube Pink,1390000.0,$1.390.000,Belleza


## 5. Extracción con paginación – categoría Capilar

Se recorren varias páginas de la categoría capilar, se consolidan todos los productos y se exporta el resultado a CSV. Entre página y página se espera 1,5 segundos para no sobrecargar el sitio.

> **Verificar antes de ejecutar:** que el patrón de la URL (`URL_PAGINACION`) corresponda a como el sitio numera sus páginas, abriendo una de ellas en el navegador.

In [ ]:
# Lista global para consolidar los productos de todas las páginas
todos_los_productos = []

# Número de páginas a descargar (de la 1 a la 12)
paginas_a_descargar = 12

# Patrón de URL de la categoría; {pagina} se reemplaza por el número de página
URL_PAGINACION = "https://www.cromantic.com/capilar{pagina}"

print("Iniciando la extracción con paginación...")

for pagina in range(1, paginas_a_descargar + 1):
    url_dinamica = URL_PAGINACION.format(pagina=pagina)
    print(f"Procesando página {pagina} de {paginas_a_descargar}... ", end="")

    try:
        response = requests.get(url_dinamica, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"Saltada por código de error: {response.status_code}")
            continue

        soup_pag = BeautifulSoup(response.content, 'html.parser')
        productos_pagina = extraer_productos(soup_pag, "Capilar, Belleza")
        todos_los_productos.extend(productos_pagina)

        print(f"¡Completada! Se encontraron {len(productos_pagina)} productos.")

    except Exception as e:
        print(f"Error al procesar la página {pagina}: {e}")

    # Esperamos 1.5 segundos entre páginas para respetar el sitio web
    time.sleep(1.5)

# Creamos el DataFrame consolidado
df_paginado = pd.DataFrame(todos_los_productos)

if df_paginado.empty:
    print("\nNo se recolectaron productos. Revisa el patrón de URL de la paginación.")
else:
    # Eliminamos productos repetidos usando la referencia (SKU)
    df_paginado = df_paginado.drop_duplicates(subset=['Referencia (SKU)'])

    print(f"\nExtracción finalizada. Total de productos únicos recolectados: {len(df_paginado)}")

    # Mostramos las primeras filas de los datos integrados
    display(df_paginado.head(15))

    # Exportamos y descargamos el archivo consolidado
    nombre_archivo_total = "productos_cromantic.csv"
    df_paginado.to_csv(nombre_archivo_total, index=False, encoding='utf-8-sig', sep=';')
    files.download(nombre_archivo_total)

Iniciando la extracción con paginación...
Procesando página 1 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 2 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 3 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 4 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 5 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 6 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 7 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 8 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 9 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 10 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 11 de 12... ¡Completada! Se encontraron 12 productos.
Procesando página 12 de 12... ¡Completada! Se encontraron 12 productos.

Extracción finalizada. Total de productos únicos recolectados: 13


,Referencia (SKU),Nombre/Marca,Valor (COP),Valor,Características
0,10603,[Blow & Bliss] Kit Capilar Essential Hair Blow...,67991.0,$67.991,"Capilar, Belleza"
1,11030,[Olaplex] Kit Olaplex Hair Hydration Heroes,165990.0,$165.990,"Capilar, Belleza"
2,11029,[Olaplex] Kit Olaplex The Key to Shine,109990.0,$109.990,"Capilar, Belleza"
3,11002,[Anyeluz] Mascarilla Terapia Crecimiento Anyel...,45990.0,$45.990,"Capilar, Belleza"
4,11015,[Everyday Love] Shampoo Everyday Love Miracle ...,49900.0,$49.900,"Capilar, Belleza"
5,11016,[Everyday Love] Acondicionados Everyday Love M...,49900.0,$49.900,"Capilar, Belleza"
6,11018,[Everyday Love] Óleo Capilar Everyday Love Mir...,44900.0,$44.900,"Capilar, Belleza"
7,11019,[Everyday Love] Shampoo Everyday Love Wavy Edi...,49900.0,$49.900,"Capilar, Belleza"
8,11020,[Everyday Love] Acondicionador Everyday Love W...,49900.0,$49.900,"Capilar, Belleza"
9,11021,[Everyday Love] Crema para Peinar Everyday Lov...,49900.0,$49.900,"Capilar, Belleza"


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Análisis descriptivo

Resumen de los datos recolectados en la categoría capilar, usando la columna numérica de precio.

In [ ]:
if not df_paginado.empty:
    # Separar la marca del nombre para poder agrupar
    df_paginado['Marca'] = df_paginado['Nombre/Marca'].str.extract(r'^\[(.*?)\]')

    print("Estadísticas de precio (COP):")
    display(df_paginado['Valor (COP)'].describe().round(0))

    print("\nCantidad de productos por marca (top 10):")
    display(df_paginado['Marca'].value_counts().head(10))

    print("\nPrecio promedio por marca (top 10, COP):")
    display(df_paginado.groupby('Marca')['Valor (COP)'].mean().sort_values(ascending=False).head(10).round(0))

    print("\nLos 5 productos más caros:")
    display(df_paginado.nlargest(5, 'Valor (COP)')[['Nombre/Marca', 'Valor']])

Estadísticas de precio (COP):


,Valor (COP)
count,13.0
mean,67256.0
std,36625.0
min,44900.0
25%,45990.0
50%,49900.0
75%,67991.0
max,165990.0



Cantidad de productos por marca (top 10):


,count
Marca,
Everyday Love,6
Anyeluz,3
Olaplex,2
Blow & Bliss,1
Loreal Professionnel,1



Precio promedio por marca (top 10, COP):


,Valor (COP)
Marca,
Olaplex,137990.0
Loreal Professionnel,99990.0
Blow & Bliss,67991.0
Everyday Love,49067.0
Anyeluz,45323.0



Los 5 productos más caros:


,Nombre/Marca,Valor
1,[Olaplex] Kit Olaplex Hair Hydration Heroes,$165.990
2,[Olaplex] Kit Olaplex The Key to Shine,$109.990
11,[Loreal Professionnel] Kit Duo Mini L'Oréal Pr...,$99.990
0,[Blow & Bliss] Kit Capilar Essential Hair Blow...,$67.991
4,[Everyday Love] Shampoo Everyday Love Miracle ...,$49.900
